# Grover’s Algorithm — Short Implementation Report  
**Prepared By: Tanmay Nair 25BCE5539**

This notebook implements Grover’s Algorithm using two approaches:  
1. A **NumPy-based simulator**  
2. A **Qiskit quantum circuit implementation**  

The goal is to demonstrate how Grover’s search amplifies the probability of marked states.

---

## Algorithm Summary
1. **Initialize Superposition**  
   All `2^n` basis states are given equal amplitude.  
   - NumPy: uniform vector  
   - Qiskit: Hadamard gates on all qubits  

2. **Oracle (Phase Flip)**  
   The target state(s) have their phase flipped from `+1` to `-1`.  
   - NumPy: modify diagonal entry  
   - Qiskit: X-gates + multi-controlled-Z  

3. **Diffuser (Inversion About the Mean)**  
   Reflects amplitudes around their average, boosting the marked state.  
   Implemented with:
   - NumPy: matrix formula  
   - Qiskit: H → X → multi-controlled-Z → X → H  

4. **Grover Iteration**  
   Repeat: **Oracle → Diffuser**  
   Optimal number of iterations is:
   \[
   r \approx \left\lfloor \frac{\pi}{4}\sqrt{N} \right\rfloor
   \]

5. **Measurement / Visualization**  
   - NumPy: shows probability evolution across iterations  
   - Qiskit: runs on Aer or IBMQ and displays measurement histogram  

---

## What the Notebook Demonstrates
- How amplitude amplification works  
- How Grover’s circuit is constructed  
- Animated probability changes using widgets  
- Realistic simulation using Qiskit Aer  

This provides an interactive, visual way to understand the core mechanics of Grover’s quantum search.


### Cell 1 — Environment Check

This cell verifies that all required packages (`numpy`, `matplotlib`, `ipywidgets`, `qiskit`, `qiskit-aer`, `qiskit-ibm-runtime`) are installed.  

If anything is missing, it prints a recommended `pip install` command.  
This ensures the notebook runs correctly regardless of the user’s environment.


In [ ]:
# --- CELL 1: Environment check (run first) ---
import sys
import importlib
missing = []
for pkg in ("numpy","matplotlib","ipywidgets","qiskit","qiskit_aer","qiskit_ibm_runtime"):
    try:
        importlib.import_module(pkg)
    except Exception:
        missing.append(pkg)
if missing:
    print("Missing packages detected:", missing)
    print("\nRecommended install command (run in your activated env / terminal):")
    print("pip install numpy matplotlib ipywidgets qiskit qiskit-aer qiskit-ibm-runtime")
    print("\nIf installation fails because of build tools, run first:")
    print("python -m pip install --upgrade pip setuptools wheel build")
else:
    print("All required packages appear installed.")
print("\nNote: If you're using VS Code's Notebook UI, ensure the kernel is the environment where you installed packages.")


: 

### Cell 2 — Imports & Backend Detection

This cell imports the main libraries used for the visualizer:
- NumPy, Matplotlib  
- ipywidgets for interactivity  
- Qiskit modules (circuit builder, Aer, visualization, IBMQ runtime)

It also checks which Qiskit components are available and prints their status so the notebook can fall back gracefully when some features (e.g., circuit drawing or Aer) are missing.


In [ ]:
# --- CELL 2: Imports & backend detection ---

%matplotlib inline

import math, time, getpass
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output, HTML


import ipywidgets as widgets

qiskit_available = True
qiskit_aer_available = True
qiskit_ibm_runtime_available = True
qiskit_viz_available = True

try:
    import qiskit
    from qiskit import transpile
except Exception:
    qiskit_available = False

try:
    from qiskit_aer import Aer
except Exception:
    qiskit_aer_available = False

try:
    from qiskit.visualization import circuit_drawer, plot_histogram
except Exception:
    qiskit_viz_available = False

try:
    from qiskit_ibm_runtime import QiskitRuntimeService, Session, Sampler
except Exception:
    qiskit_ibm_runtime_available = False

print("qiskit:", qiskit_available, " qiskit-aer:", qiskit_aer_available,
      " qiskit-viz:", qiskit_viz_available, " qiskit-ibm-runtime:", qiskit_ibm_runtime_available)


### Cell 3 — NumPy Grover Simulator

This cell implements Grover’s Algorithm **purely with NumPy**, without Qiskit.

It includes:
- Creating a uniform state  
- Building the oracle matrix  
- Building the diffusion matrix  
- Combining them into a Grover step  
- Recording the full state evolution (`history`)

This lightweight simulator is used for the probability animation and for users who may not have Qiskit installed.


In [ ]:
# --- CELL 3: Core NumPy Grover simulator (state history) ---
def create_uniform_state(n): 
    N = 2**n
    return np.ones(N, dtype=complex) / math.sqrt(N)

def oracle_matrix(n, marked_indices): 
    N = 2**n
    O = np.eye(N, dtype=complex)
    for idx in marked_indices:
        if 0 <= idx < N:
            O[idx, idx] = -1
    return O

def diffusion_matrix(n): 
    N = 2**n
    psi = np.ones((N, N), dtype=complex) / N
    return 2*psi - np.eye(N, dtype=complex)

def grover_step(state, O, D):
    return D @ (O @ state)

def run_grover_numpy(n, marked_indices, iterations): 
    state = create_uniform_state(n)
    O = oracle_matrix(n, marked_indices)
    D = diffusion_matrix(n)
    history = [state.copy()]
    for _ in range(iterations):
        state = grover_step(state, O, D)
        history.append(state.copy())
    return history

def suggested_optimal_iterations(n, m):
    N = 2**n
    if m <= 0: return 1
    return max(1, int(math.floor((math.pi/4) * math.sqrt(N / m))))


### Cell 4 — Qiskit Grover Circuit Builder

This cell constructs a full Grover circuit using Qiskit:
- Applies Hadamards  
- Builds the oracle for any set of marked states  
- Builds the diffuser  
- Adds measurements  

Also includes `show_circuit_inline()` which displays the circuit using Qiskit’s visual tools or falls back to text if needed.


In [ ]:
# --- CELL 4: Qiskit Grover circuit builder & draw helper ---
def build_grover_qc(n, marked_indices):
    if not qiskit_available:
        return None
    from qiskit import QuantumCircuit
    qc = QuantumCircuit(n, n, name=f"Grover(n={n})")
    qc.h(range(n))
    
    for idx in marked_indices:
        bits = format(idx, f'0{n}b')[::-1]  
        for q, bit in enumerate(bits):
            if bit == '0':
                qc.x(q)
        if n == 1:
            qc.z(0)
        else:
            try:
                qc.mct(list(range(n-1)), n-1)
                qc.z(n-1)
                qc.mct(list(range(n-1)), n-1)
            except Exception:
                
                qc.h(n-1)
                for i in range(n-1):
                    qc.cx(i, n-1)
                qc.z(n-1)
                for i in reversed(range(n-1)):
                    qc.cx(i, n-1)
                qc.h(n-1)
        for q, bit in enumerate(bits):
            if bit == '0':
                qc.x(q)
    
    qc.barrier()
    qc.h(range(n))
    qc.x(range(n))
    if n == 1:
        qc.z(0)
    else:
        try:
            qc.mct(list(range(n-1)), n-1)
            qc.z(n-1)
            qc.mct(list(range(n-1)), n-1)
        except Exception:
            qc.h(n-1)
            for i in range(n-1):
                qc.cx(i, n-1)
            qc.z(n-1)
            for i in reversed(range(n-1)):
                qc.cx(i, n-1)
            qc.h(n-1)
    qc.x(range(n))
    qc.h(range(n))
    qc.barrier()
    qc.measure(range(n), range(n))
    return qc

def show_circuit_inline(n, marked_indices):
    if qiskit_available and qiskit_viz_available:
        qc = build_grover_qc(n, marked_indices)
        try:
            
            fig = circuit_drawer(qc, output='mpl', idle_wires=False)
            display(fig)
            return
        except Exception:
           
            display(Markdown("**Circuit (text fallback):**"))
            display(Markdown("```\n" + qc.draw(output='text') + "\n```"))
            return
    
    display(Markdown("**Circuit (ASCII fallback):**"))
    desc = ["Qubits: " + ", ".join(f"q{i}" for i in range(n)),
            "Init: H on all qubits -> uniform superposition"]
    for idx in marked_indices:
        desc.append(f"Oracle: flip phase of |{format(idx, f'0{n}b')}>")
    desc.append("Diffusion: H, X, multi-ctrl-Z, X, H")
    display(Markdown("```\n" + "\n".join(desc) + "\n```"))


### Cell 5 — Aer Run Helper

This cell defines a helper function to run a Grover circuit on the **Aer simulator** using Qiskit's modern `.run()` API.  
It transpiles the circuit and returns measurement counts.  
Used when the user chooses "Run on Aer".


In [ ]:
# --- CELL 5: Qiskit Aer run helper (modern .run API) ---
def run_qc_on_aer(qc, shots=1024):
    if not qiskit_aer_available:
        raise RuntimeError("qiskit-aer not installed")
    # Using Aer.get_backend('aer_simulator') and .run()
    backend = Aer.get_backend('aer_simulator')
    t_qc = transpile(qc, backend)
    job = backend.run(t_qc, shots=shots)
    result = job.result()
    return result.get_counts()


### Cell 6 — (Duplicate) Aer Helper

This cell appears to be a duplicate of the Aer helper above and is functionally similar.  
It ensures compatibility with Qiskit’s newer API and backend loading.


In [ ]:
# --- CELL 6: Qiskit Aer run helper (modern .run API) ---
def run_qc_on_aer(qc, shots=1024):
    if not qiskit_aer_available:
        raise RuntimeError("qiskit-aer not installed")
    # Using Aer.get_backend('aer_simulator') and .run()
    backend = Aer.get_backend('aer_simulator')
    t_qc = transpile(qc, backend)
    job = backend.run(t_qc, shots=shots)
    result = job.result()

### Cell 7 — Probability Plot & Animation Tools

This cell provides:
- A bar-plot function for showing probability distributions  
- An interactive animation widget (`Play`, `Slider`)  
- A frame-by-frame display of the state evolution from the NumPy simulator  

It powers the animated probability viewer seen in the interactive UI.


In [ ]:
# --- CELL 7: Visualization functions: probability bar & animation using Play widget ---
def plot_probs(state, ax=None, title=None):
    probs = np.abs(state)**2
    N = len(probs)
    x = np.arange(N)
    if ax is None:
        fig, ax = plt.subplots(figsize=(8,3))
    ax.clear()
    ax.bar(x, probs)
    ax.set_xlabel("Basis state (decimal)")
    ax.set_ylabel("Probability")
    ax.set_xticks(x)
    if title: ax.set_title(title)
    plt.tight_layout()


def display_history_with_player(history, autoplay=False, interval_ms=400):
    if not history:
        display(Markdown("No history to show"))
        return
    n_frames = len(history)
    play = widgets.Play(value=0, min=0, max=n_frames-1, step=1, description="Play")
    speed = widgets.IntSlider(value=interval_ms, min=100, max=2000, step=100, description="ms")
    frame_slider = widgets.IntSlider(value=0, min=0, max=n_frames-1, step=1, description='Frame')
    widgets.jslink((play, 'value'), (frame_slider, 'value'))
    
    out = widgets.Output()
    box = widgets.VBox([widgets.HBox([play, speed]), frame_slider, out])
    display(box)
    
    with out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8,3))
        plot_probs(history[0], ax=ax, title=f'Frame 0')
        display(fig)
    
    def on_frame_change(change):
        idx = change['new']
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(8,3))
            plot_probs(history[idx], ax=ax, title=f'Frame {idx}')
            display(fig)
    frame_slider.observe(on_frame_change, names='value')
    
    if autoplay:
        play.value = 0


### Cell 8 — Interactive Controls (Widgets)

This cell builds the full interactive panel:
- Sliders for selecting number of qubits and iterations  
- Text box for choosing marked states  
- Buttons for:
  - Running NumPy simulation  
  - Running on Aer  
  - Viewing the circuit  
  - Running full animation  
  - Submitting jobs to IBMQ Runtime  
- Random target generator  
- Output display area  

All buttons are wired to their callbacks, enabling users to explore Grover’s Algorithm visually and interactively.


In [ ]:
# --- CELL 8: Controls (interactive widget panel) ---

n_slider = widgets.IntSlider(value=3, min=1, max=6, step=1, description='Qubits (n)')
iters_slider = widgets.IntSlider(value=1, min=1, max=20, step=1, description='Iterations')
marked_text = widgets.Text(value='1', placeholder='e.g. 3 or 1,6', description='Marked (csv)')
random_btn = widgets.Button(description='Random target(s)')
run_btn = widgets.Button(description='Run (NumPy)')
run_qbtn = widgets.Button(description='Run on Aer (Qiskit)')
run_and_show_btn = widgets.Button(description='Run + Show circuit + Animate')
ibmq_token_text = widgets.Text(value='', placeholder='(optional) IBMQ token', description='IBMQ token:')
ibmq_run_btn = widgets.Button(description='Run on IBMQ (runtime)')
output = widgets.Output(layout={'border': '1px solid gray'})

def parse_marked(s, n):
    s = s.strip()
    if s == '':
        return []
    N = 2**n
    parts = [p.strip() for p in s.split(',') if p.strip()!='']
    res = []
    for p in parts:
        try:
            idx = int(p)
            if 0 <= idx < N:
                res.append(idx)
        except:
            pass
    return sorted(list(set(res)))

import random
def random_clicked(b):
    n = n_slider.value
    N = 2**n
    k = min(N, max(1, int(round(N/8))))
    k = min(k, 3)
    chosen = random.sample(range(N), k)
    marked_text.value = ','.join(str(x) for x in chosen)

random_btn.on_click(random_clicked)

def run_numpy_clicked(b):
    with output:
        clear_output()
        n = n_slider.value
        iters = iters_slider.value
        marked = parse_marked(marked_text.value, n)
        if not marked:
            print("No marked indices provided, picking one random.")
            marked = [random.randint(0, 2**n - 1)]
            print("Picked:", marked)
        print(f"Running NumPy Grover: n={n}, marked={marked}, iterations={iters}")
        history = run_grover_numpy(n, marked, iters)
        display_history_with_player(history)
        print("Done (NumPy).")

run_btn.on_click(run_numpy_clicked)

def run_aer_clicked(b):
    with output:
        clear_output()
        if not qiskit_aer_available:
            print("qiskit-aer not available. Install with pip install qiskit-aer")
            return
        n = n_slider.value
        marked = parse_marked(marked_text.value, n)
        if not marked:
            print("No marked indices: pick random.")
            marked = [random.randint(0, 2**n - 1)]
            print("Picked:", marked)
        qc = build_grover_qc(n, marked)
        if qc is None:
            print("Qiskit not available to build circuit.")
            return
        print("Constructed Qiskit circuit. Running on Aer (simulator)...")
        try:
            counts = run_qc_on_aer(qc, shots=1024)
            display(Markdown("**Counts:**"))
            display(counts)
            if qiskit_viz_available:
                try:
                    display(plot_histogram(counts))
                except Exception:
                    print("Could not show plot_histogram.")
        except Exception as e:
            print("Aer run error:", e)

run_qbtn.on_click(run_aer_clicked)

def run_and_show_clicked(b):
    with output:
        clear_output()
        n = n_slider.value
        iters = iters_slider.value
        marked = parse_marked(marked_text.value, n)
        if not marked:
            print("No marked indices: picking random")
            marked = [random.randint(0, 2**n - 1)]
            print("Picked:", marked)
        print(f"NumPy simulate and animate: n={n}, marked={marked}, iterations={iters}")
        history = run_grover_numpy(n, marked, iters)
        display_history_with_player(history, autoplay=False)
        print("\nCircuit diagram:")
        show_circuit_inline(n, marked)

run_and_show_btn.on_click(run_and_show_clicked)

def run_ibmq_clicked(b):
    with output:
        clear_output()
        if not qiskit_ibm_runtime_available:
            print("qiskit-ibm-runtime not installed. Install with pip install qiskit-ibm-runtime")
            return
        n = n_slider.value
        marked = parse_marked(marked_text.value, n)
        if not marked:
            print("No marked indices: pick random.")
            marked = [random.randint(0, 2**n - 1)]
            print("Picked:", marked)
        qc = build_grover_qc(n, marked)
        token = ibmq_token_text.value.strip() or None
        try:
            job = run_qc_on_ibmq_runtime(qc, token=token, shots=8192)
            display(Markdown(f"Submitted IBMQ job: {job}"))
            print("Monitor via IBM Quantum dashboard or service APIs.")
        except Exception as e:
            print("IBMQ runtime error:", e)

ibmq_run_btn.on_click(run_ibmq_clicked)


controls_row1 = widgets.HBox([n_slider, iters_slider, marked_text, random_btn])
controls_row2 = widgets.HBox([run_btn, run_qbtn, run_and_show_btn])
controls_row3 = widgets.HBox([ibmq_token_text, ibmq_run_btn])
display(controls_row1, controls_row2, controls_row3, output)
